# Attention Pooling

Replaces `mean`/`max` pooling with a content-aware weighted sum over a sequence.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

## Option 1 — Linear query (projection-based)

In [ ]:
class AttentionPooling(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.query = nn.Linear(d_model, 1)

    def forward(self, x):
        # x: (batch, seq_len, d_model)
        weights = self.query(x)                  # (batch, seq_len, 1)
        weights = F.softmax(weights, dim=1)      # normalize over seq_len
        pooled = (weights * x).sum(dim=1)        # (batch, d_model)
        return pooled

# Quick test
batch, seq_len, d_model = 2, 10, 64
x = torch.randn(batch, seq_len, d_model)
pool = AttentionPooling(d_model)
out = pool(x)
print(out.shape)  # (2, 64)

## Option 2 — Learnable query vector (dot-product style)

In [ ]:
class AttentionPoolingDot(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.query = nn.Parameter(torch.randn(d_model))

    def forward(self, x):
        # x: (batch, seq_len, d_model)
        scores = torch.einsum('bsd,d->bs', x, self.query)  # (batch, seq_len)
        weights = F.softmax(scores, dim=-1).unsqueeze(-1)   # (batch, seq_len, 1)
        return (weights * x).sum(dim=1)                     # (batch, d_model)

# Quick test
pool_dot = AttentionPoolingDot(d_model)
out_dot = pool_dot(x)
print(out_dot.shape)  # (2, 64)